# Imports General

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OrdinalEncoder, MinMaxScaler, OneHotEncoder

# Carga de Datos

In [2]:
archivo1 = "clientes_banca_raw.csv"
df = pd.read_csv(archivo1)
df = df.drop(['id_cliente'], axis=1)
df = df.drop(['salario_mensual_aprox'], axis=1)
df = df.drop(['comentarios_cliente'], axis=1)
df = df.drop(['motivo_cancelacion'], axis=1)

# Partición Train / Test

In [3]:
X = df.drop('abandono', axis=1)
Y = df['abandono']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Tabla Resumen Auditoria Inicial

| Variable | Tipo de característica | Poder Predictivo Observado | Decisión Técnica |
| :--- | :--- | :--- | :--- |
| **id_cliente** | Identificador | Ninguna porque solo es un código administrativo | Quitar la columna porque solo identifica al cliente y no sirve para aprender. |
| **edad** | Numérica continua | Casi nula ya que fieles y abandonos tienen edades muy similares | Borrar la edad imposible de 999 años, rellenar los huecos vacíos con la mediana y guardarla por si sirve por grupos de edad. |
| **ingresos_anuales** | Numérica continua | Muy débil de forma directa | Rellenar los huecos vacíos con la mediana, revisar los que tienen 0 € y retocar los sueldos más altos. |
| **salario_mensual_aprox** | Numérica continua | Totalmente repetida con los ingresos anuales | Quitar la columna porque dice exactamente lo mismo que los ingresos anuales. |
| **antiguedad_meses** | Numérica discreta | Los clientes más antiguos tienden a quedarse algo más | Rellenar las 5 filas vacías con la mediana de la antigüedad para no perder clientes. |
| **saldo_cuenta** | Numérica continua | Clave porque los clientes con más saldo casi siempre se quedan | No hay que rellenar nada y solo ajustar los saldos más ricos para no descompensar. |
| **nivel_estudios** | Categórica ordinal | Apenas influye en la decisión de marcharse | Rellenar los huecos vacíos con Grado y pasar los estudios a números ordenados de menor a mayor. |
| **tipo_cuenta** | Categórica nominal | Influye bastante ya que Ahorro pierde un 54% de clientes mientras que Inversión solo pierde un 32% | Arreglar faltas y espacios, cambiar el signo de interrogación por Corriente y convertir cada cuenta en columnas separadas. |
| **nivel_satisfaccion** | Categórica ordinal | Muy decisiva ya que el 92% de los muy descontentos cancela la cuenta | Guardar con máxima prioridad y pasar las respuestas a números del 1 al 5. |
| **num_productos** | Numérica discreta | Influye claramente porque cuantos más productos tienen contratados menos se van | No hay que tocar nada porque los números del 1 al 4 están correctos. |
| **tiene_tarjeta** | Categórica binaria | No influye porque la proporción de tarjetas es idéntica en ambos grupos | No hay que tocar nada porque ya viene lista con ceros y unos. |
| **canal_preferido** | Categórica nominal | Muestra pequeñas diferencias según la vía de contacto elegida | No hay que rellenar nada y poner cada canal en una columna distinta. |
| **comentarios_cliente** | Categórica nominal | No se puede usar porque casi todas las filas están vacías | Quitar la columna del modelo. |
| **motivo_cancelacion** | Categórica nominal | Es trampa porque solo lo rellenan los que ya se han ido | Quitar la columna para no falsear los resultados. |
| **abandono** | Categórica binaria | Es la variable objetivo y representa lo que queremos predecir | No hay que tocar nada porque está completa, equilibrada y lista con ceros y unos. |

# Fase 4 - Preprocesamiento: Numérica

## Sanitización de centinelas

In [4]:
# Columnas numéricas con valores centinelas
cols_con_centinelas = ['edad', 'ingresos_anuales', 'saldo_cuenta']

# Reemplazamos centinelas conocidos por np.nan de forma explícita en Train y Test
X_train.loc[(X_train['edad'] <= 0) | (X_train['edad'] > 120), 'edad'] = np.nan
X_train.loc[X_train['ingresos_anuales'] <= 0, 'ingresos_anuales'] = np.nan
X_train.loc[X_train['saldo_cuenta'] == -999, 'saldo_cuenta'] = np.nan

X_test.loc[(X_test['edad'] <= 0) | (X_test['edad'] > 120), 'edad'] = np.nan
X_test.loc[X_test['ingresos_anuales'] <= 0, 'ingresos_anuales'] = np.nan
X_test.loc[X_test['saldo_cuenta'] == -999, 'saldo_cuenta'] = np.nan

print("\nNulos detectados en X_train tras sustituir centinelas:")
print(X_train[cols_con_centinelas].isna().sum())

print("\nNulos detectados en X_test tras sustituir centinelas:")
print(X_test[cols_con_centinelas].isna().sum())


Nulos detectados en X_train tras sustituir centinelas:
edad                 9
ingresos_anuales    10
saldo_cuenta         0
dtype: int64

Nulos detectados en X_test tras sustituir centinelas:
edad                2
ingresos_anuales    0
saldo_cuenta        0
dtype: int64


## Imputación numérica de los valores nulos

In [5]:
# Agrupamos las características numéricas
cols_mediana = ['ingresos_anuales', 'antiguedad_meses', 'saldo_cuenta']
cols_media = ['edad']

# Imputador por mediana
imputer_mediana = SimpleImputer(strategy='median')
imputer_mediana.fit(X_train[cols_mediana])

X_train[cols_mediana] = imputer_mediana.transform(X_train[cols_mediana])
X_test[cols_mediana] = imputer_mediana.transform(X_test[cols_mediana])

# Imputador por media
imputer_media = SimpleImputer(strategy='mean')
imputer_media.fit(X_train[cols_media])

X_train[cols_media] = imputer_media.transform(X_train[cols_media])
X_test[cols_media] = imputer_media.transform(X_test[cols_media])

# Inspección de los parámetros aprendidos
print("Medianas aprendidas por columna:")
for col, val in zip(cols_mediana, imputer_mediana.statistics_):
    print(f" - {col}: {val:.2f}")

print("\nMedia aprendida por columna:")
for col, val in zip(cols_media, imputer_media.statistics_):
    print(f" - {col}: {val:.2f}")

Medianas aprendidas por columna:
 - ingresos_anuales: 27167.64
 - antiguedad_meses: 61.00
 - saldo_cuenta: 5810.57

Media aprendida por columna:
 - edad: 42.09


In [6]:
# Comprueba que no quedan nulos ni en X_train ni en X_test
cols_imputadas = cols_mediana + cols_media

print("Nulos en variables numericas de X_train:")
print(X_train[cols_imputadas].isna().sum())

print("\nNulos en variables numericas de X_test:")
print(X_test[cols_imputadas].isna().sum())

Nulos en variables numericas de X_train:
ingresos_anuales    0
antiguedad_meses    0
saldo_cuenta        0
edad                0
dtype: int64

Nulos en variables numericas de X_test:
ingresos_anuales    0
antiguedad_meses    0
saldo_cuenta        0
edad                0
dtype: int64


## Escalado numérico según la naturaleza de la variable

**Variables con StandardScaler**

Voy a usar StandardScaler para **edad** y **antiguedad_meses** porque:
* En la **edad**, una vez que quito el dato de 999 años, los números son normales y van de 18 a 78 años, sin valores raros que desvíen la media.
* En la **antigüedad**, los clientes están repartidos de forma equilibrada entre 1 y 120 meses, sin casos extremos.
Como no tengo valores exagerados que engañen a la media, uso StandardScaler porque es perfecto para ponerlas en la misma escala.

**Variables con RobustScaler**

Voy a usar RobustScaler para **ingresos_anuales** y **saldo_cuenta** porque:
* En ambas columnas tengo clientes con muchísimo dinero, llegando a sueldos de hasta 345.000 € y saldos de casi 239.000 €.
* Esos números son reales de clientes con mucho patrimonio en el banco, no son errores, así que no los puedo borrar.
Si usara la media, esos pocos clientes estirarían toda la escala y darían errores al modelo. Por eso elijo RobustScaler, que escala fijándose en la mediana en vez sin tener en cuenta los extremos, así me quito que esos sueldos y saldos altos descompensen la escala.

**StandardScaler**

In [7]:
# Columnas simétricas y sin outliers reales
cols_standard = ['edad', 'antiguedad_meses']

scaler_std = StandardScaler()
scaler_std.fit(X_train[cols_standard])

print("Parámetros aprendidos por StandardScaler:")
for col, media, desv in zip(cols_standard, scaler_std.mean_, scaler_std.scale_):
    print(f" - {col}: media (μ) = {media:.2f} | desviación (σ) = {desv:.2f}")

# Transformamos Train y Test aprendiendo solo de Train
X_train[cols_standard] = scaler_std.transform(X_train[cols_standard])
X_test[cols_standard] = scaler_std.transform(X_test[cols_standard])

Parámetros aprendidos por StandardScaler:
 - edad: media (μ) = 42.09 | desviación (σ) = 11.59
 - antiguedad_meses: media (μ) = 61.00 | desviación (σ) = 32.60


**RobustScaler**

In [8]:
# Columnas con asimetría y outliers reales de negocio
cols_robust = ['ingresos_anuales', 'saldo_cuenta']

scaler_rob = RobustScaler()
scaler_rob.fit(X_train[cols_robust])

print("Parámetros aprendidos por RobustScaler:")
for col, med, iqr in zip(cols_robust, scaler_rob.center_, scaler_rob.scale_):
    print(f" - {col}: mediana = {med:.2f} | IQR = {iqr:.2f}")

# Transformamos Train y Test aprendiendo solo de Train
X_train[cols_robust] = scaler_rob.transform(X_train[cols_robust])
X_test[cols_robust] = scaler_rob.transform(X_test[cols_robust])

Parámetros aprendidos por RobustScaler:
 - ingresos_anuales: mediana = 27167.64 | IQR = 16136.70
 - saldo_cuenta: mediana = 5810.57 | IQR = 14527.35


## Auditoría final


In [9]:
# Primeras 10 filas de X_train para comprobar que las variables numericas estan escaladas
X_train.head(10)

,edad,ingresos_anuales,antiguedad_meses,saldo_cuenta,nivel_estudios,tipo_cuenta,nivel_satisfaccion,num_productos,tiene_tarjeta,canal_preferido
145,0.251078,-0.589291,-1.012232,-0.292502,Secundaria,Nomina,Medio,2,1,Oficina
9,0.078565,1.730610,0.000064,0.313117,Grado,Ahorro,Medio,2,0,App Movil
375,0.251078,-0.244450,0.981684,-0.053864,Grado,Ahorro,Alto,2,0,App Movil
523,-1.474055,-0.773509,-0.858854,1.396014,Secundaria,Corriente,Muy Bajo,3,1,App Movil
188,0.337335,-0.416831,-1.257637,3.330132,Doctorado,Nomina,Muy Alto,3,1,Oficina
131,0.251078,-0.124322,-0.674800,7.217833,Grado,Corriente,Medio,2,0,App Movil
572,-0.352718,3.251926,-1.656420,-0.183432,Grado,Corriente,Medio,1,1,App Movil
298,1.889955,-0.776304,1.104386,1.383126,Secundaria,Nomina,Muy Bajo,2,0,App Movil
586,-1.474055,-0.442480,-0.460070,-0.399974,Secundaria,Nomina,Medio,2,1,Oficina
446,-0.266462,-1.073769,-0.521422,0.421347,Secundaria,Ahorro,Medio,3,0,App Movil


In [10]:
# Primeras 10 filas de X_test para comprobar que las variables numericas estan escaladas
X_test.head(10)

,edad,ingresos_anuales,antiguedad_meses,saldo_cuenta,nivel_estudios,tipo_cuenta,nivel_satisfaccion,num_productos,tiene_tarjeta,canal_preferido
110,0.509848,-0.104868,-1.257637,1.369122,Grado,Corriente,Medio,1,1,App Movil
419,-0.352718,-0.147841,-1.257637,3.381987,Master,Ahorro,Bajo,2,0,Telefono
565,-1.646569,0.585563,-0.766827,0.074383,Grado,Corriente,Medio,1,1,Web
77,-0.611488,-0.411595,0.306820,0.889287,NaN,Corriente,Medio,2,0,Web
181,-1.560312,1.147978,1.809926,0.440748,Primaria,Ahorro,Bajo,2,1,App Movil
284,-2.077852,-0.340242,1.533845,0.040529,Primaria,Nomina,Medio,1,1,Oficina
10,0.164822,0.173597,0.674928,-0.172240,Master,corriente,Bajo,2,1,App Movil
469,-0.438975,0.008224,-1.595068,-0.399974,Secundaria,Corriente,Medio,2,1,Web
78,0.000000,1.222056,-0.705475,-0.030694,Grado,Nomina,Medio,1,1,App Movil
349,-0.870259,0.359542,-0.061287,1.063119,Secundaria,Ahorro,Bajo,2,1,App Movil


In [11]:
# Parametros aprendidos por StandardScaler
print("Parametros StandardScaler:")
print("Media:", scaler_std.mean_)
print("Escala:", scaler_std.scale_)

# Parametros aprendidos por RobustScaler
print("\nParametros RobustScaler:")
print("Centro:", scaler_rob.center_)
print("Escala:", scaler_rob.scale_)

Parametros StandardScaler:
Media: [42.08917197 60.99791667]
Escala: [11.59330399 32.59917554]

Parametros RobustScaler:
Centro: [27167.64   5810.565]
Escala: [16136.7025 14527.3475]


**Por qué X_test no tiene exactamente media 0 ni mediana 0**

He ajustado los escaladores aprendiendo las medidas con los clientes de X_train. Cuando luego transformo X_test, uso esa misma regla aprendida en el entrenamiento para no hacer trampa con datos del futuro.

Como los clientes del grupo de prueba son personas diferentes a los del entrenamiento, sus sueldos y edades cambian un poco. Por eso, al aplicarles la regla en X_train, sus resultados quedan muy cerca de 0 pero no dan 0 exacto. Esto es lo correcto porque simula cómo se comportará mi modelo en la vida real con clientes nuevos.

# Fase 5 - Preprocesamiento: Categórica

## Sanitización y Normalización de Centinelas

In [12]:
X_train['tipo_cuenta'] = X_train['tipo_cuenta'].replace('?', np.nan)
X_test['tipo_cuenta'] = X_test['tipo_cuenta'].replace('?', np.nan)

X_train['tipo_cuenta'] = X_train['tipo_cuenta'].str.strip().str.capitalize()
X_test['tipo_cuenta'] = X_test['tipo_cuenta'].str.strip().str.capitalize()

## Imputación cualitativa

In [13]:
# Imputamos las columnas categoricas con la moda
cols_categoricas = ['tipo_cuenta', 'canal_preferido', 'nivel_estudios', 'nivel_satisfaccion']

imputer_cat = SimpleImputer(strategy='most_frequent')
imputer_cat.fit(X_train[cols_categoricas])

print("Modas aprendidas en X_train:")
for col, moda in zip(cols_categoricas, imputer_cat.statistics_):
    print(f" - {col}: '{moda}'")

X_train[cols_categoricas] = imputer_cat.transform(X_train[cols_categoricas])
X_test[cols_categoricas] = imputer_cat.transform(X_test[cols_categoricas])

Modas aprendidas en X_train:
 - tipo_cuenta: 'Corriente'
 - canal_preferido: 'App Movil'
 - nivel_estudios: 'Grado'
 - nivel_satisfaccion: 'Medio'


In [14]:
# Verifica que no quedan nulos en dichas columnas
print("Nulos en variables categóricas de X_train:")
print(X_train[cols_categoricas].isna().sum())

print("\nNulos en variables categóricas de X_test:")
print(X_test[cols_categoricas].isna().sum())

Nulos en variables categóricas de X_train:
tipo_cuenta           0
canal_preferido       0
nivel_estudios        0
nivel_satisfaccion    0
dtype: int64

Nulos en variables categóricas de X_test:
tipo_cuenta           0
canal_preferido       0
nivel_estudios        0
nivel_satisfaccion    0
dtype: int64


## Codificación Ordinal y Escalado

In [15]:
# Definimos el orden de menor a mayor y aplicamos OrdinalEncoder
orden_estudios = ['Primaria', 'Secundaria', 'Grado', 'Master', 'Doctorado']
orden_satisfaccion = ['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto']

encoder_ordinal = OrdinalEncoder(categories=[orden_estudios, orden_satisfaccion])

cols_ordinales = ['nivel_estudios', 'nivel_satisfaccion']
encoder_ordinal.fit(X_train[cols_ordinales])

X_train_ord = encoder_ordinal.transform(X_train[cols_ordinales])
X_test_ord = encoder_ordinal.transform(X_test[cols_ordinales])

In [16]:
# Escalamos las variables ordinales entre 0 y 1 con MinMaxScaler
scaler_ord = MinMaxScaler()
X_train_ord_scaled = scaler_ord.fit_transform(X_train_ord)
X_test_ord_scaled = scaler_ord.transform(X_test_ord)

In [17]:
# Resultado despues de escalar
print("X_train_ord_scaled:")
print(X_train_ord_scaled)

print("X_test_ord_scaled:")
print(X_test_ord_scaled)

X_train_ord_scaled:
[[0.25 0.5 ]
 [0.5  0.5 ]
 [0.5  0.75]
 [0.25 0.  ]
 [1.   1.  ]
 [0.5  0.5 ]
 [0.5  0.5 ]
 [0.25 0.  ]
 [0.25 0.5 ]
 [0.25 0.5 ]
 [0.25 0.5 ]
 [0.25 0.25]
 [0.5  1.  ]
 [0.5  0.5 ]
 [0.5  0.  ]
 [0.25 1.  ]
 [0.25 0.75]
 [0.5  0.5 ]
 [0.25 0.75]
 [0.   0.5 ]
 [0.75 0.5 ]
 [0.25 0.5 ]
 [0.75 0.25]
 [0.25 0.75]
 [0.75 0.5 ]
 [0.25 0.  ]
 [0.   0.75]
 [0.5  0.25]
 [0.5  0.5 ]
 [0.5  0.5 ]
 [0.5  0.75]
 [0.25 0.25]
 [0.25 0.25]
 [0.5  0.75]
 [0.5  0.75]
 [0.5  0.5 ]
 [1.   0.  ]
 [0.25 0.75]
 [0.5  0.25]
 [0.5  0.25]
 [0.25 1.  ]
 [0.25 0.5 ]
 [0.5  0.25]
 [0.75 0.75]
 [0.5  0.  ]
 [0.5  0.5 ]
 [0.5  0.75]
 [0.25 0.75]
 [0.5  0.  ]
 [0.25 0.25]
 [0.5  1.  ]
 [0.   0.75]
 [0.5  1.  ]
 [0.25 0.75]
 [0.5  0.25]
 [0.25 0.25]
 [0.25 0.75]
 [0.25 0.75]
 [0.25 0.  ]
 [0.75 0.  ]
 [0.   0.75]
 [0.25 1.  ]
 [0.25 0.5 ]
 [1.   0.5 ]
 [0.5  0.  ]
 [0.25 1.  ]
 [0.5  0.5 ]
 [0.25 0.25]
 [0.   0.75]
 [0.75 0.75]
 [0.75 0.5 ]
 [0.5  0.5 ]
 [0.75 0.5 ]
 [0.   0.  ]
 [0.5  0.5 ]
 [0.2

## Codificación nominal

In [18]:
# Codificamos las variables nominales con OneHotEncoder
cols_nominales = ['tipo_cuenta', 'canal_preferido']

encoder_ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

encoder_ohe.fit(X_train[cols_nominales])

X_train_ohe = encoder_ohe.transform(X_train[cols_nominales])
X_test_ohe = encoder_ohe.transform(X_test[cols_nominales])

nombres_dummies = encoder_ohe.get_feature_names_out(cols_nominales)
print(f"Número de características dummy generadas: {len(nombres_dummies)}")
print("Nuevas columnas generadas:")
for nombre in nombres_dummies:
    print(f" - {nombre}")

Número de características dummy generadas: 9
Nuevas columnas generadas:
 - tipo_cuenta_Ahorro
 - tipo_cuenta_Corriente
 - tipo_cuenta_Inversion
 - tipo_cuenta_Nomina
 - canal_preferido_App Movil
 - canal_preferido_Cajero Antiguo
 - canal_preferido_Oficina
 - canal_preferido_Telefono
 - canal_preferido_Web


## Ensamble Manual y Auditoría Dimensional

In [19]:
# Unimos las ramas numerica, ordinal escalada y nominal con np.hstack
cols_numericas = ['edad', 'ingresos_anuales', 'antiguedad_meses', 'saldo_cuenta', 'num_productos', 'tiene_tarjeta']

nombres_columnas_final = (
    cols_numericas +
    cols_ordinales +
    list(encoder_ohe.get_feature_names_out(cols_nominales))
)

X_train_final = pd.DataFrame(
    np.hstack([
        X_train[cols_numericas].to_numpy(),
        X_train_ord_scaled,
        X_train_ohe
    ]),
    columns=nombres_columnas_final,
    index=X_train.index
)

X_test_final = pd.DataFrame(
    np.hstack([
        X_test[cols_numericas].to_numpy(),
        X_test_ord_scaled,
        X_test_ohe
    ]),
    columns=nombres_columnas_final,
    index=X_test.index
)

print(f"Dimensiones de X_train_final: {X_train_final.shape}")
print(f"Dimensiones de X_test_final:  {X_test_final.shape}")

display(X_train_final.head(10))
display(X_test_final.head(10))

Dimensiones de X_train_final: (480, 17)
Dimensiones de X_test_final:  (120, 17)


,edad,ingresos_anuales,antiguedad_meses,saldo_cuenta,num_productos,tiene_tarjeta,nivel_estudios,nivel_satisfaccion,tipo_cuenta_Ahorro,tipo_cuenta_Corriente,tipo_cuenta_Inversion,tipo_cuenta_Nomina,canal_preferido_App Movil,canal_preferido_Cajero Antiguo,canal_preferido_Oficina,canal_preferido_Telefono,canal_preferido_Web
145,0.251078,-0.589291,-1.012232,-0.292502,2.0,1.0,0.25,0.50,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
9,0.078565,1.730610,0.000064,0.313117,2.0,0.0,0.50,0.50,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
375,0.251078,-0.244450,0.981684,-0.053864,2.0,0.0,0.50,0.75,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
523,-1.474055,-0.773509,-0.858854,1.396014,3.0,1.0,0.25,0.00,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
188,0.337335,-0.416831,-1.257637,3.330132,3.0,1.0,1.00,1.00,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
131,0.251078,-0.124322,-0.674800,7.217833,2.0,0.0,0.50,0.50,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
572,-0.352718,3.251926,-1.656420,-0.183432,1.0,1.0,0.50,0.50,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
298,1.889955,-0.776304,1.104386,1.383126,2.0,0.0,0.25,0.00,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
586,-1.474055,-0.442480,-0.460070,-0.399974,2.0,1.0,0.25,0.50,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
446,-0.266462,-1.073769,-0.521422,0.421347,3.0,0.0,0.25,0.50,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


,edad,ingresos_anuales,antiguedad_meses,saldo_cuenta,num_productos,tiene_tarjeta,nivel_estudios,nivel_satisfaccion,tipo_cuenta_Ahorro,tipo_cuenta_Corriente,tipo_cuenta_Inversion,tipo_cuenta_Nomina,canal_preferido_App Movil,canal_preferido_Cajero Antiguo,canal_preferido_Oficina,canal_preferido_Telefono,canal_preferido_Web
110,0.509848,-0.104868,-1.257637,1.369122,1.0,1.0,0.50,0.50,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
419,-0.352718,-0.147841,-1.257637,3.381987,2.0,0.0,0.75,0.25,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
565,-1.646569,0.585563,-0.766827,0.074383,1.0,1.0,0.50,0.50,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
77,-0.611488,-0.411595,0.306820,0.889287,2.0,0.0,0.50,0.50,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
181,-1.560312,1.147978,1.809926,0.440748,2.0,1.0,0.00,0.25,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
284,-2.077852,-0.340242,1.533845,0.040529,1.0,1.0,0.00,0.50,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
10,0.164822,0.173597,0.674928,-0.172240,2.0,1.0,0.75,0.25,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
469,-0.438975,0.008224,-1.595068,-0.399974,2.0,1.0,0.25,0.50,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
78,0.000000,1.222056,-0.705475,-0.030694,1.0,1.0,0.50,0.50,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
349,-0.870259,0.359542,-0.061287,1.063119,2.0,1.0,0.25,0.25,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
